# Chapter `2.1` - Model Context Protocol

### Importing the necessary libraries

In [1]:
# Base utils.
import sys
from os import getenv
from pathlib import Path
from dotenv import load_dotenv

# Response formatting
from pprint import pprint
from IPython.display import Markdown

# Model init. and invocation
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

# MCP server
from langchain_mcp_adapters.client import MultiServerMCPClient
import langchain_mcp_adapters.sessions as lc_sessions
import mcp.client.stdio as mcp_stdio

In [2]:
load_dotenv()

GOOGLE_API_KEY = getenv("GOOGLE_API_KEY")
GEMINI_API_MODEL = getenv("GEMINI_API_MODEL")

### Reqired m/c setup for MCP server

In [3]:
MCP_ERRLOG = open(r"./logs/mcp_stderr.log", "a", encoding="utf-8", buffering=1)
_orig_stdio_client = mcp_stdio.stdio_client

def _patched_stdio_client(server, errlog=None):
    return _orig_stdio_client(server, errlog=MCP_ERRLOG if errlog is None else errlog)


# NOTE: Patch both references used by adapters
mcp_stdio.stdio_client = _patched_stdio_client
lc_sessions.stdio_client = _patched_stdio_client # type: ignore

print("Patch active:", lc_sessions.stdio_client is _patched_stdio_client) # type: ignore

Patch active: True


## MCP Server

### 1. Local

#### Creating MCP client for local MCP server

In [5]:
server_script = Path.cwd() / "resources" / "mcp_server.py"
print(f'MCP server {"exists" if server_script.exists() else "NOT found"} at: {server_script}')

MCP server exists at: d:\GitHub\lca-lc-foundations\notebooks\module-2\resources\mcp_server.py


In [6]:
client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": sys.executable,
            "args": [str(server_script)],
            "cwd": str(server_script.parent),
        }
    }
)

tools = await client.get_tools()
resources = await client.get_resources("local_server")
prompt_messages = await client.get_prompt("local_server", "prompt")

print(prompt_messages[0].content if prompt_messages else "No prompt")


    You are a helpful assistant that answers user questions about LangChain, LangGraph and LangSmith.

    You can use the following tools/resources to answer user questions:
    - search_web: Search the web for latest available information.
    - github_file: Access the langchain-ai repo files for obtaining information from the official GitHub repo.

    Rules to be taken into consideration while responding to user queries:
    1. If the user asks a question that is not related to LangChain, LangGraph or LangSmith, you should say "Sorry, I can only answer questions related to LangChain, LangGraph and LangSmith."
    2. You may try multiple tool and resource calls to respond to the user's query.
    3. You may also ask clarifying questions to the user to get a better understanding of their query.
    


#### Initializing the agent

In [7]:
def _msg_to_text(m: BaseMessage) -> str:
    if isinstance(m.content, str):
        return m.content

    return "\n".join(part if isinstance(part, str) else str(part) for part in m.content)

system_prompt_text = "\n\n".join(_msg_to_text(m) for m in prompt_messages).strip()

In [8]:
model = ChatGoogleGenerativeAI(model=GEMINI_API_MODEL, api_key=GOOGLE_API_KEY)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt_text
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [9]:
config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Give me whatever information you have about the langchain-mcp-adapters library.")]},
    config=config # type: ignore
)

In [24]:
Markdown(response['messages'][-1].content)

The `langchain-mcp-adapters` library is a lightweight wrapper that makes Anthropic Model Context Protocol (MCP) tools compatible with LangChain and LangGraph. It bridges LangChain's agent framework with MCP's standardized tool ecosystem, allowing developers to integrate hundreds of existing MCP servers without writing custom adapters.

Here are some key features and benefits:

*   **Seamless Integration:** Converts MCP tools into LangChain- and LangGraph-compatible tools.
*   **Multi-Server Support:** Enables interaction with tools across multiple MCP servers simultaneously.
*   **Simplified Development:** Eliminates the complexity of manual tool integration.
*   **Content Handling:** Converts multi-part content (like text and images) from MCP servers into LangChain's standard content blocks.
*   **Resource and Prompt Conversion:** Provides adapters for converting MCP resources to LangChain Blob objects and MCP prompts to LangChain messages.
*   **Secure Connections:** Supports OAuth 2.0 authentication for secure MCP servers.

This library is particularly useful for agents that need to pull from multiple MCP servers at once, enabling the combination of different tools for more powerful applications.

### 2. Online

In [25]:
client_online = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client_online.get_tools()

In [27]:
for tool in tools:
    pprint(dict(tool))

{'args_schema': {'properties': {'timezone': {'description': 'IANA timezone '
                                                            'name (e.g., '
                                                            "'America/New_York', "
                                                            "'Europe/London'). "
                                                            'Use '
                                                            "'America/New_York' "
                                                            'as local timezone '
                                                            'if no timezone '
                                                            'provided by the '
                                                            'user.',
                                             'type': 'string'}},
                 'required': ['timezone'],
                 'type': 'object'},
 'callbacks': None,
 'coroutine': <function convert_mcp_tool_to_langchain_tool.<loca

In [28]:
agent_online_mcp = create_agent(
    model=model,
    tools=tools,
)

In [29]:
question = HumanMessage(content="What time is it in Volgograd?")
response = await agent_online_mcp.ainvoke({"messages": [question]})

In [31]:
Markdown(response['messages'][-1].content)

The time in Volgograd is 5:26 PM on Sunday.